In [0]:
##aws_utils
import boto3
from botocore.exceptions import NoCredentialsError
 
from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime
 
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)
 
# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:  
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
 
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    
 
# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']
 
# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  
 
    env = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env

env=get_env_prefix()

run_date_input = dbutils.widgets.get("run_date").strip()

if run_date_input:
    ref_date = run_date_input
else:
    ref_date = spark.sql(f"""
        SELECT MAX(BusinessDate) FROM `{env}-cdh`.primary.exposure WHERE SourceSystem = 'profits'
    """).collect()[0][0].strftime("%Y-%m-%d")

target_schema = f"`{env}-cdh-delivery`.it"
perimeter_schema = f"`{env}-cdh`.derivativeinternal"

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

# Helper function: write temp view to Delta table (partition overwrite by BusinessDate)
def write_npv_table(view_name, table_name):
    spark.sql(f"SELECT *, CAST('{ref_date}' AS DATE) AS BusinessDate FROM {view_name}") \
        .write.format("delta") \
        .option("delta.columnMapping.mode", "name") \
        .option("delta.minReaderVersion", "2") \
        .option("delta.minWriterVersion", "5") \
        .option("encoding", "UTF-8") \
        .mode("overwrite") \
        .option("replaceWhere", f"BusinessDate = '{ref_date}'") \
        .saveAsTable(f"{target_schema}.{table_name}")

print(f"ref_date = {ref_date}")
print(f"env = {env}")

In [0]:
# =============================================================================
# Step 0b - Pre-cache Base Tables
# Creates temp views for all source tables, fully filtered by ref_date.
# All downstream steps reference ONLY these views.
# =============================================================================

# ALL_EXPOSURES: active + closed loans (for perimeter + CLT_REL)
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW all_exposures AS
    SELECT 
        ExposureKey,
        ExposureSourceSystemKey,
        AgreementKey,
        CDI,
        PartyName,
        PartyRelationshipType,
        PrimaryPartyFlag,
        PortfolioKey,
        Opendate,
        Defaultdate,
        PurposeKT
    FROM `{env}-cdh`.primary.get_all_exposures('{ref_date}')
    WHERE SourceSystem = 'profits'
      AND PortfolioKey IN ('7', '11', '5', '6', '10')
""")

# EXPOSURE: active loans only (for NPV_ACCOUNTS financials/buckets)
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW exposure AS
    SELECT 
        ExposureKey, ExposureSourceSystemKey, AgreementKey,
        PortfolioKey, PrimaryPartyFlag, IsActiveFlag,
        ExposureStatus, Currency, BucketCode,
        Opendate, Maturitydate,
        Principal, interest, Unrealizedinterest, Expense,
        Penaltyinterest, PenaltyUnrealizedinterest,
        OffBalance, OutstandingTotalBalance,
        DueAmount, Dueinterest, DueUnrealizedinterest, DueAccrued,
        InterestRate, InstalmentFrequency, InterestFrequency,
        NextInstallmentdate, NextInstallmentAmount,
        GracePeriodEnddate, GracePeriodType, MechanismCode, SlaveFlag
    FROM `{env}-cdh`.primary.exposure
    WHERE SourceSystem = 'profits'
      AND BusinessDate = '{ref_date}'
      AND PortfolioKey IN ('7', '11', '5', '6', '10')
""")

# PARTY
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW party AS
    SELECT 
        CDI, TaxRegistrationNumber, LegalNameGR,
        GroupCode, GroupName,
        EconomicActivityCode, EconomicActivity,
        CreditRatingTypeCode, MonitoringUnitCode
    FROM `{env}-cdh`.primaryactive.party
    WHERE BusinessDate = '{ref_date}'
      AND CDI IS NOT NULL
""")

# PARTY_VALUATOR: valuator names for real estate
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW party_valuator AS
    SELECT PartyKey, LastNameGR AS ValuatorName
    FROM `{env}-cdh`.primaryactive.party
    WHERE BusinessDate = '{ref_date}'
      AND SourceSystem = 'thesis'
""")

# COLLATERAL
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW collateral AS
    SELECT 
        CollateralId, CollateralCode, CollateralOldKey,
        AgreementKey, RealEstateKey,
        Category, SubCategoryCode, SubCategory,
        CollateralAmount, CollateralCurrency,
        InitialLegalOrder, OwnershipStatus, OwnerPartyKey
    FROM `{env}-cdh`.primaryactive.collateral
    WHERE BusinessDate = '{ref_date}'
""")

# REALESTATEBUILDING
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW realestatebuilding AS
    SELECT 
        RealEstateKey, RealEstateCepalKey,
        RealEstateAddress1, RealEstateAddress2, RealEstateCity,
        RealEstatePostalCode, RealEstateDistrict,
        RealEstateMarketValue, RealEstateEvaluationDate,
        RealEstateEvaluationType,
        RealEstateForcedSaleValue,
        RealEstateLandArea,
        RealEstateConstructionYear,
        RealEstateUsageCodeTypeCode, RealEstateUsageCodeType,
        RealEstateAuctionAmount, RealEstateAuctionDate
    FROM `{env}-cdh`.primaryactive.realestatebuilding
    WHERE BusinessDate = '{ref_date}'
      AND RealEstateCepalKey IS NOT NULL
""")

# REALESTATEVALUATION
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW realestatevaluation AS
    SELECT 
        RealEstateKey, RealEstateValuationKey, RealEstateValuatorKey, ValuationDate
    FROM `{env}-cdh`.primaryactive.realestatevaluation
    WHERE BusinessDate = '{ref_date}'
""")

# PORTFOLIO
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW portfolio AS
    SELECT PortfolioKey, unit_code_profits
    FROM `{env}-cdh`.primaryactive.portfolio
    WHERE BusinessDate = '{ref_date}'
""")

# LOOKUPS
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW bhfmr AS
    SELECT DISTINCT SERIAL_NUM, DESCRIPTION
    FROM `{env}-cdh`.landed.profits_params
    WHERE PARAM_TYPE = 'BHFMR' AND SERIAL_NUM > 0
""")

spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW monitoring_unit AS
    SELECT DISTINCT Code, Descpription AS Description
    FROM `{env}-cdh`.landed.params_monitoringunit
""")

print("All base views created.")

In [0]:
# =============================================================================
# Step 1 - Build Perimeter
# Builds the loan/party perimeter using get_all_exposures (active + closed loans).
# Portfolio groups: primary (7, 11) + secondary (5, 6, 10 - only parties in primary).
# Output: v_perimeter_final (materialized to {perimeter_schema}.npv_perimeter)
# =============================================================================

# Step 1a: Primary perimeter - Portfolios 7, 11 (Galaxy IV + Cosmos Wholesale)
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_per1 AS
    SELECT DISTINCT
        e.AgreementKey                                                          AS AgreementNumber,
        e.CDI                                                                   AS PartyId,
        e.PartyName,
        e.PartyRelationshipType                                                 AS RelationType,
        CASE WHEN e.PartyRelationshipType = 'Primary Owner' AND e.PrimaryPartyFlag = true 
             THEN '1' ELSE '0' END                                             AS MainBenefFlag,
        CASE WHEN e.PartyRelationshipType = 'Primary Owner' AND e.PrimaryPartyFlag = true 
             THEN 'A1D' ELSE 'STH' END                                         AS IndPel,
        'Y'                                                                     AS AirBRqdF,
        e.Opendate                                                              AS AgreementIssueDate
    FROM all_exposures e
    WHERE e.PortfolioKey IN ('7', '11')
      AND e.PartyRelationshipType IN ('Primary Owner', 'Guarantor', 'Co-Owner')
""")

# Step 1b: Secondary perimeter - Portfolios 5, 6, 10 (only parties already in primary)
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_perimeter AS
    SELECT 
        AgreementNumber, PartyId, PartyName, RelationType,
        MainBenefFlag, IndPel, AirBRqdF, AgreementIssueDate,
        CAST(NULL AS DATE) AS MinLoanOpenDt,
        CAST(NULL AS DATE) AS MinDefaultDt
    FROM v_per1

    UNION

    SELECT DISTINCT
        e.AgreementKey                                                          AS AgreementNumber,
        e.CDI                                                                   AS PartyId,
        e.PartyName,
        e.PartyRelationshipType                                                 AS RelationType,
        CASE WHEN e.PartyRelationshipType = 'Primary Owner' AND e.PrimaryPartyFlag = true 
             THEN '1' ELSE '0' END                                             AS MainBenefFlag,
        CASE WHEN e.PartyRelationshipType = 'Primary Owner' AND e.PrimaryPartyFlag = true 
             THEN 'A1D' ELSE 'STH' END                                         AS IndPel,
        'Y'                                                                     AS AirBRqdF,
        e.Opendate                                                              AS AgreementIssueDate,
        CAST(NULL AS DATE)                                                      AS MinLoanOpenDt,
        CAST(NULL AS DATE)                                                      AS MinDefaultDt
    FROM all_exposures e
    WHERE e.PortfolioKey IN ('5', '6', '10')
      AND e.PartyRelationshipType IN ('Primary Owner', 'Guarantor', 'Co-Owner')
      AND EXISTS (SELECT 1 FROM v_per1 p WHERE p.PartyId = e.CDI)
""")

# Step 1c: Final perimeter with AirBRqdF override and aggregates
spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_perimeter_build AS
    SELECT 
        p.AgreementNumber,
        p.PartyId,
        p.PartyName,
        p.RelationType,
        p.MainBenefFlag,
        p.IndPel,
        CASE 
            WHEN p.AgreementNumber IN (
                SELECT DISTINCT e2.AgreementKey
                FROM all_exposures e2
                WHERE e2.PurposeKT IN ('ΚΑΤΑΝΑΛΩΤΙΚΑ ΔΑΝΕΙΑ', 'ΣΤΕΓΑΣΤΙΚΑ ΔΑΝΕΙΑ')
            ) THEN 'N'
            ELSE p.AirBRqdF
        END AS AirBRqdF,
        p.AgreementIssueDate,
        agg.MinLoanOpenDt,
        agg.MinDefaultDt
    FROM v_perimeter p
    LEFT JOIN (
        SELECT 
            e3.AgreementKey,
            MIN(e3.Opendate) AS MinLoanOpenDt,
            MIN(e3.Defaultdate) AS MinDefaultDt
        FROM all_exposures e3
        WHERE e3.PrimaryPartyFlag = true
        GROUP BY e3.AgreementKey
    ) agg ON agg.AgreementKey = p.AgreementNumber
""")

# Materialize perimeter to Delta table
spark.sql(f"SELECT *, CAST('{ref_date}' AS DATE) AS BusinessDate FROM v_perimeter_build") \
    .write.format("delta") \
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .option("encoding", "UTF-8") \
    .mode("overwrite") \
    .option("replaceWhere", f"BusinessDate = '{ref_date}'") \
    .saveAsTable(f"{perimeter_schema}.npv_perimeter")

# Read back from materialized table as temp view for downstream steps
spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW v_perimeter_final AS
    SELECT * FROM {perimeter_schema}.npv_perimeter WHERE BusinessDate = '{ref_date}'
""")

In [0]:
# =============================================================================
# Step 2 - NPV_CUSTOMER
# Source: party + v_perimeter_final + bhfmr + monitoring_unit
# Output: {target_schema}.npv_customer
# =============================================================================

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_npv_customer AS
    WITH party_rank AS (
        SELECT 
            PartyId,
            IndPel,
            AgreementIssueDate,
            ROW_NUMBER() OVER (PARTITION BY PartyId ORDER BY IndPel, AgreementNumber) AS rnk
        FROM v_perimeter_final
    ),
    party_agg AS (
        SELECT 
            PartyId,
            MIN(MinLoanOpenDt) AS MinLoanOpenDt,
            MIN(MinDefaultDt) AS MinDefaultDt
        FROM v_perimeter_final
        GROUP BY PartyId
    ),
    party_dedup AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY CDI 
                ORDER BY 
                    CASE WHEN TaxRegistrationNumber IS NOT NULL THEN 0 ELSE 1 END,
                    CASE WHEN EconomicActivityCode IS NOT NULL THEN 0 ELSE 1 END,
                    CASE WHEN GroupCode IS NOT NULL THEN 0 ELSE 1 END
            ) AS party_rnk
        FROM party
    )
    SELECT
        COALESCE(pty.TaxRegistrationNumber, 'no-afm')                          AS OBLG_TAX_NUM,
        pty.CDI                                                                 AS OBLG_CDI_CODE,
        COALESCE(pty.GroupCode, '?')                                            AS OBLG_GRP_CODE,
        pty.LegalNameGR                                                         AS OBLG_RGST_NM,
        COALESCE(pty.GroupName, pty.LegalNameGR)                                AS GRP_NM,
        COALESCE(pty.EconomicActivityCode, '?')                                 AS STAKOD_LVL_3_NM,
        NULL                                                                    AS CNCTD_BRWR_CNCTD_F,
        NULL                                                                    AS FIN_REF_YEAR,
        NULL                                                                    AS OBLG_FNC_TVR,
        NULL                                                                    AS OBLG_RTG_FNL_RTG_CODE,
        NULL                                                                    AS OBLG_RTG_FNL_RTG_DT,
        COALESCE(CAST(pty.CreditRatingTypeCode AS STRING), '?')                AS OBLG_RLTNP_UNIT_CODE,
        COALESCE(bhfmr.DESCRIPTION, '?')                                       AS OBLG_RLTNP_UNIT_DSC,
        CASE WHEN COALESCE(pty.MonitoringUnitCode, '?') IN ('N/A', '0') THEN '?' 
             ELSE COALESCE(pty.MonitoringUnitCode, '?') END                     AS OBLG_ARS_MGT_UNIT_CODE,
        CASE WHEN COALESCE(pty.MonitoringUnitCode, '?') IN ('N/A', '0') THEN '?' 
             ELSE COALESCE(mu.Description, pty.MonitoringUnitCode, '?') END     AS OBLG_ARS_MGT_UNIT_DSC,
        NULL                                                                    AS SVC_BR_LVL_3_BR_CODE,
        NULL                                                                    AS SVC_BR_LVL_3_BR_NM,
        COALESCE(agg.MinLoanOpenDt, pr.AgreementIssueDate)                      AS OBLG_BSN_RLTNP_STRT_DT,
        agg.MinDefaultDt                                                        AS OBLG_CHG_IN_ARS_MGT_UNIT_DT,
        COALESCE(pty.EconomicActivity, '?')                                     AS STAKOD_LVL_3_DSC,
        pr.IndPel                                                               AS IND_PEL
    FROM party_dedup pty
    JOIN party_rank pr ON pr.PartyId = pty.CDI AND pr.rnk = 1
    LEFT JOIN party_agg agg ON agg.PartyId = pty.CDI
    LEFT JOIN bhfmr ON bhfmr.SERIAL_NUM = pty.CreditRatingTypeCode
    LEFT JOIN monitoring_unit mu ON mu.Code = pty.MonitoringUnitCode
    WHERE pty.party_rnk = 1
""")

write_npv_table("v_npv_customer", "npv_customer")

In [0]:
# =============================================================================
# Step 3 - NPV_ACCOUNTS
# Source: exposure + portfolio + v_perimeter_final
# One row per active loan (PrimaryPartyFlag, IsActiveFlag, SlaveFlag=false).
# Output: {target_schema}.npv_accounts
# =============================================================================

spark.sql(f"""
    CREATE OR REPLACE TEMPORARY VIEW v_npv_accounts AS
    WITH agreement_info AS (
        SELECT DISTINCT
            e.AgreementKey,
            pf.IndPel,
            pf.AirBRqdF,
            port.unit_code_profits,
            ROW_NUMBER() OVER (PARTITION BY e.AgreementKey ORDER BY pf.IndPel, pf.AirBRqdF DESC) AS rnk
        FROM exposure e
        JOIN (
            SELECT DISTINCT AgreementNumber, IndPel, AirBRqdF,
                   RANK() OVER (PARTITION BY AgreementNumber ORDER BY IndPel, AirBRqdF DESC) AS rnk
            FROM v_perimeter_final
        ) pf ON pf.AgreementNumber = e.AgreementKey AND pf.rnk = 1
        JOIN portfolio port ON port.PortfolioKey = e.PortfolioKey
        WHERE e.PrimaryPartyFlag = true
          AND e.IsActiveFlag = true
    ),
    main_benef AS (
        SELECT DISTINCT AgreementNumber, PartyId AS CDI
        FROM v_perimeter_final
        WHERE MainBenefFlag = '1'
    )
    SELECT
        e.ExposureKey                                                           AS AC_CODE,
        e.ExposureSourceSystemKey                                               AS AC_ID,
        mb.CDI                                                                  AS AC_OBLG_CDI_CODE,
        e.AgreementKey                                                          AS AC_CTR_ID,
        e.AgreementKey                                                          AS AC_CTR_CODE,
        'PRF'                                                                   AS AC_CTR_SRC_STM_CODE,
        agr.AirBRqdF                                                            AS AIRB_RQD_F,
        'PRF'                                                                   AS AC_SRC_STM_CODE,
        e.Principal                                                             AS AC_FIN_DTL_PNP,
        e.Currency                                                              AS AC_FIN_DTL_PNP_CCY_CODE,
        COALESCE(e.interest, 0) + COALESCE(e.Penaltyinterest, 0)               AS AC_FIN_DTL_INT,
        e.Currency                                                              AS AC_FIN_DTL_INT_CCY_CODE,
        COALESCE(e.Expense, 0)                                                  AS AC_FIN_DTL_FEE,
        e.Currency                                                              AS AC_FIN_DTL_FEE_CCY_CODE,
        COALESCE(e.Unrealizedinterest, 0) + COALESCE(e.PenaltyUnrealizedinterest, 0) AS AC_FIN_DTL_SUSP_INT,
        e.Currency                                                              AS AC_FIN_DTL_SUSP_INT_CCY_CODE,
        e.OffBalance                                                            AS AC_FIN_DTL_OFF_BAL_SHET_BAL,
        e.Currency                                                              AS AC_FIN_DTL_OFF_BAL_SHET_BAL_CCY_CODE,
        NULL                                                                    AS AC_MPRM_FAIR_VAL_ADJ,
        NULL                                                                    AS AC_MPRM_BAL,
        NULL                                                                    AS AC_MPRM_BAL_CCY_CODE,
        NULL                                                                    AS AC_MPRM_BAL_IN_EUR,
        NULL                                                                    AS AC_MPRM_TP_CODE,
        NULL                                                                    AS AC_MPRM_TP_DSC,
        NULL                                                                    AS TOT_PRVN,
        CASE WHEN e.BucketCode = '1'  THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_1_AC_BAL,
        CASE WHEN e.BucketCode = '1'  THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_1_SUSP_INT,
        CASE WHEN e.BucketCode = '2'  THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_2_AC_BAL,
        CASE WHEN e.BucketCode = '2'  THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_2_SUSP_INT,
        CASE WHEN e.BucketCode = '3'  THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_3_AC_BAL,
        CASE WHEN e.BucketCode = '3'  THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_3_SUSP_INT,
        CASE WHEN e.BucketCode = '4'  THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_4_AC_BAL,
        CASE WHEN e.BucketCode = '4'  THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_4_SUSP_INT,
        CASE WHEN e.BucketCode = '5'  THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_5_AC_BAL,
        CASE WHEN e.BucketCode = '5'  THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_5_SUSP_INT,
        CASE WHEN e.BucketCode = '6'  THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_6_AC_BAL,
        CASE WHEN e.BucketCode = '6'  THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_6_SUSP_INT,
        CASE WHEN e.BucketCode = '7'  THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_7_AC_BAL,
        CASE WHEN e.BucketCode = '7'  THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_7_SUSP_INT,
        CASE WHEN e.BucketCode = '8'  THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_8_AC_BAL,
        CASE WHEN e.BucketCode = '8'  THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_8_SUSP_INT,
        CASE WHEN e.BucketCode = '9'  THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_9_AC_BAL,
        CASE WHEN e.BucketCode = '9'  THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_9_SUSP_INT,
        CASE WHEN e.BucketCode = '10' THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_10_AC_BAL,
        CASE WHEN e.BucketCode = '10' THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_10_SUSP_INT,
        CASE WHEN e.BucketCode = '11' THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_11_AC_BAL,
        CASE WHEN e.BucketCode = '11' THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_11_SUSP_INT,
        CASE WHEN e.BucketCode = '12' THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_12_AC_BAL,
        CASE WHEN e.BucketCode = '12' THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_12_SUSP_INT,
        CASE WHEN e.BucketCode = '13' THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_13_AC_BAL,
        CASE WHEN e.BucketCode = '13' THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_13_SUSP_INT,
        CASE WHEN e.BucketCode = '14' THEN CAST(e.DueAmount AS DECIMAL(19,2)) END AS BKT_ANL_BKT_14_AC_BAL,
        CASE WHEN e.BucketCode = '14' THEN COALESCE(e.Dueinterest,0) + COALESCE(e.DueUnrealizedinterest,0) + COALESCE(e.DueAccrued,0) END AS BKT_ANL_BKT_14_SUSP_INT,
        MONTHS_BETWEEN(e.Maturitydate, e.Opendate)                              AS AC_DRTN,
        COALESCE(e.InstalmentFrequency, '?')                                    AS AC_INSTMT_FRQ_CODE,
        COALESCE(e.InterestFrequency, '?')                                      AS AC_INT_FRQ_CODE,
        e.Opendate                                                              AS AC_APRV_DT,
        CASE WHEN e.ExposureStatus IN ('Active', 'Performing') 
             THEN e.NextInstallmentdate END                                     AS AC_PYMT_PLN_PYMT_DT_CAPITAL,
        CASE WHEN e.ExposureStatus IN ('Active', 'Performing') 
                  AND e.MechanismCode IN ('4', '6') 
             THEN e.NextInstallmentdate END                                     AS AC_PYMT_PLN_PYMT_DT_INTEREST,
        CASE WHEN e.ExposureStatus IN ('Active', 'Performing') 
             THEN e.NextInstallmentAmount END                                   AS AC_PYMT_PLN_PNP,
        NULL                                                                    AS AC_PYMT_PLN_INT,
        CASE WHEN e.GracePeriodEnddate > '{ref_date}' THEN 1 ELSE 0 END            AS AC_GRC_PRD_IND,
        CASE WHEN e.GracePeriodEnddate > '{ref_date}' THEN e.GracePeriodType END    AS AC_GRC_PRD_TP_CODE,
        CASE WHEN e.GracePeriodEnddate > '{ref_date}' THEN e.Opendate END           AS AC_GRC_PRD_STRT_DT,
        CASE WHEN e.GracePeriodEnddate > '{ref_date}' THEN e.GracePeriodEnddate END AS AC_GRC_PRD_END_DT,
        e.OutstandingTotalBalance                                               AS AC_FIN_DTL_AC_BAL,
        COALESCE(e.InterestRate, 0)                                             AS AC_CRN_NML_INT_RATE,
        agr.IndPel                                                              AS IND_PEL
    FROM exposure e
    JOIN agreement_info agr ON agr.AgreementKey = e.AgreementKey AND agr.rnk = 1
    JOIN main_benef mb ON mb.AgreementNumber = e.AgreementKey
    LEFT JOIN portfolio port ON port.PortfolioKey = e.PortfolioKey
    WHERE e.IsActiveFlag = true
      AND e.PrimaryPartyFlag = true
      AND e.SlaveFlag = false
      AND EXISTS (SELECT 1 FROM v_perimeter_final pf WHERE pf.AgreementNumber = e.AgreementKey)
""")

write_npv_table("v_npv_accounts", "npv_accounts")

In [0]:
# =============================================================================
# Step 4 - Collateral Tables (GNL, GNT, PDC)
# Source: collateral + v_perimeter_final
# Output: {target_schema}.npv_clt_gnl / npv_clt_gnt / npv_clt_pdc
# =============================================================================

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_collateral AS
    SELECT DISTINCT
        c.CollateralId,
        c.CollateralOldKey                                                      AS ReferenceCode,
        c.CollateralCurrency,
        c.SubCategoryCode,
        c.SubCategory                                                           AS SubCategoryDescription,
        COALESCE(pf.IndPel, 'UNKW')                                             AS IND_PEL,
        c.AgreementKey                                                          AS AgreementNumber,
        c.CollateralAmount
    FROM collateral c
    JOIN (
        SELECT DISTINCT AgreementNumber, IndPel,
               RANK() OVER (PARTITION BY AgreementNumber ORDER BY IndPel) AS rnk
        FROM v_perimeter_final
    ) pf ON pf.AgreementNumber = c.AgreementKey AND pf.rnk = 1
""")

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_npv_clt_gnl AS
    SELECT DISTINCT
        ReferenceCode                                                           AS CLT_CODE,
        'THS'                                                                   AS CLT_SRC_STM_CODE,
        CollateralCurrency                                                      AS CLT_CCY_CODE,
        CollateralAmount                                                        AS CLT_AMT,
        IND_PEL
    FROM v_collateral
""")

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_npv_clt_gnt AS
    SELECT DISTINCT
        ReferenceCode                                                           AS CLT_GNT_CLT_CODE,
        'THS'                                                                   AS CLT_GNT_CLT_SRC_STM_CODE,
        IND_PEL
    FROM v_collateral
    WHERE SubCategoryCode IN ('CSPQ01', 'CSPQ02')
""")

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_npv_clt_pdc AS
    SELECT DISTINCT
        ReferenceCode                                                           AS CLT_PDC_CLT_CODE,
        'THS'                                                                   AS CLT_PDC_CLT_SRC_STM_CODE,
        NULL                                                                    AS CLT_PDC_ST_TP_CODE,
        IND_PEL
    FROM v_collateral
    WHERE SubCategoryCode IN ('CSCH01', 'CSCH02')
""")

write_npv_table("v_npv_clt_gnl", "npv_clt_gnl")
write_npv_table("v_npv_clt_gnt", "npv_clt_gnt")
write_npv_table("v_npv_clt_pdc", "npv_clt_pdc")

In [0]:
# =============================================================================
# Step 5 - NPV_CLT_REAL_EST
# Source: collateral + realestatebuilding + realestatevaluation + party_valuator
# One row per property (dedup at RealEstateKey level).
# Output: {target_schema}.npv_clt_real_est
# =============================================================================

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_re_perimeter AS
    SELECT DISTINCT
        c.RealEstateKey,
        c.IND_PEL,
        RANK() OVER (PARTITION BY c.RealEstateKey ORDER BY c.IND_PEL) AS rnk
    FROM (
        SELECT 
            clt.RealEstateKey,
            COALESCE(pf.IndPel, 'UNKW') AS IND_PEL
        FROM collateral clt
        JOIN (
            SELECT DISTINCT AgreementNumber, IndPel,
                   RANK() OVER (PARTITION BY AgreementNumber ORDER BY IndPel) AS rnk
            FROM v_perimeter_final
        ) pf ON pf.AgreementNumber = clt.AgreementKey AND pf.rnk = 1
        WHERE clt.RealEstateKey IS NOT NULL
    ) c
""")

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_npv_clt_real_est AS
    WITH latest_valuation AS (
        SELECT 
            rv.RealEstateKey,
            rv.RealEstateValuationKey,
            rv.RealEstateValuatorKey,
            ROW_NUMBER() OVER (
                PARTITION BY rv.RealEstateKey 
                ORDER BY rv.ValuationDate DESC NULLS LAST, rv.RealEstateValuationKey DESC
            ) AS rnk
        FROM realestatevaluation rv
    ),
    ranked_re AS (
        SELECT
            CAST(c.RealEstateKey AS STRING)                                     AS CLT_REAL_EST_ID,
            reb.RealEstateCepalKey                                              AS CLT_REAL_EST_CODE,
            reb.RealEstateMarketValue                                           AS CLT_ORIG_VAL_AMT,
            c.OwnershipStatus                                                   AS CLT_REAL_EST_OWN_DSC,
            CAST(c.OwnerPartyKey AS STRING)                                     AS CLT_REAL_EST_OWNER_CDI_CODE,
            c.Category                                                          AS CLT_REAL_EST_SRC_REAL_PTY_TP_DSC,
            c.SubCategory                                                       AS CLT_REAL_EST_TP_DSC,
            CONCAT_WS(' ', TRIM(reb.RealEstateAddress1), NULLIF(TRIM(reb.RealEstateAddress2), ''), NULLIF(TRIM(reb.RealEstateCity), ''))
                                                                                AS CLT_REAL_EST_ADR,
            reb.RealEstatePostalCode                                            AS CLT_REAL_EST_PST_CODE,
            reb.RealEstateDistrict                                              AS CLT_REAL_EST_RGON,
            reb.RealEstateMarketValue                                           AS CLT_REAL_EST_LTST_VAL_AMT_RAW,
            reb.RealEstateEvaluationDate                                        AS CLT_REAL_EST_LTST_VAL_DT,
            reb.RealEstateEvaluationType                                        AS CLT_REAL_EST_EVAL_TYPE,
            reb.RealEstateForcedSaleValue                                       AS CLT_REAL_EST_RLZN_VAL,
            reb.RealEstateUsageCodeTypeCode                                     AS CLT_REAL_EST_USE_CODE,
            reb.RealEstateUsageCodeType                                         AS CLT_REAL_EST_USE_DSC,
            reb.RealEstateAuctionAmount                                         AS CLT_REAL_EST_AUCTION_AMT_RAW,
            CAST(reb.RealEstateAuctionDate AS DATE)                             AS CLT_REAL_EST_AUCTION_DT_RAW,
            reb.RealEstateLandArea                                              AS PTY_SZ_TOTAL,
            CASE WHEN CAST(reb.RealEstateConstructionYear AS INT) >= 1500
                 THEN YEAR(CURRENT_DATE()) - CAST(reb.RealEstateConstructionYear AS INT)
                 ELSE NULL END                                                  AS CLT_REAL_EST_AGE,
            CASE WHEN CAST(reb.RealEstateConstructionYear AS INT) >= 1500
                 THEN reb.RealEstateConstructionYear
                 ELSE NULL END                                                  AS CLT_REAL_EST_CNSTR_YEAR,
            pv.ValuatorName                                                     AS CLT_REAL_EST_VAL_AGNT_RAW,
            lv.RealEstateValuationKey                                           AS CLT_REAL_EST_VAL_CODE_RAW,
            re.IND_PEL,
            ROW_NUMBER() OVER (
                PARTITION BY c.RealEstateKey 
                ORDER BY c.CollateralAmount DESC NULLS LAST, c.CollateralCode
            ) AS rnk
        FROM collateral c
        JOIN v_re_perimeter re ON re.RealEstateKey = c.RealEstateKey AND re.rnk = 1
        LEFT JOIN realestatebuilding reb 
            ON reb.RealEstateKey = CAST(c.RealEstateKey AS STRING)
        LEFT JOIN latest_valuation lv
            ON lv.RealEstateKey = CAST(c.RealEstateKey AS STRING) AND lv.rnk = 1
        LEFT JOIN party_valuator pv
            ON pv.PartyKey = lv.RealEstateValuatorKey
        WHERE c.RealEstateKey IS NOT NULL
    )
    SELECT
        CLT_REAL_EST_ID,
        CLT_REAL_EST_CODE,
        CLT_ORIG_VAL_AMT,
        'EUR'                                                                   AS CLT_ORIG_VAL_CCY_CODE,
        'ΕΥΡΩ'                                                                  AS CLT_ORIG_VAL_CCY_DSC,
        CAST(CLT_REAL_EST_LTST_VAL_DT AS DATE)                                  AS CLT_ORIG_VAL_DT,
        '-100'                                                                  AS CLT_ORIG_VAL_TP_CODE,
        COALESCE(CLT_REAL_EST_EVAL_TYPE, '?')                                   AS CLT_ORIG_VAL_TP_DSC,
        CLT_REAL_EST_ADR,
        CLT_REAL_EST_AGE,
        CLT_REAL_EST_AUCTION_AMT_RAW                                            AS CLT_REAL_EST_AUCTION_AMT,
        NULL                                                                    AS CLT_REAL_EST_AUCTION_BNK_EXPN,
        NULL                                                                    AS CLT_REAL_EST_AUCTION_BOARD_AMT,
        NULL                                                                    AS CLT_REAL_EST_AUCTION_BOARD_ISSU_DT,
        CLT_REAL_EST_AUCTION_DT_RAW                                             AS CLT_REAL_EST_AUCTION_DT,
        NULL                                                                    AS CLT_REAL_EST_AUCTION_EXPN,
        'THS'                                                                   AS CLT_REAL_EST_CLT_SRC_STM_CODE,
        'THESIS'                                                                AS CLT_REAL_EST_CLT_SRC_STM_NM,
        CLT_REAL_EST_CNSTR_YEAR                                                 AS CLT_REAL_EST_CNSTR_DT,
        NULL                                                                    AS CLT_REAL_EST_CONSENTING_SALE_F,
        NULL                                                                    AS CLT_REAL_EST_CTRY_ID,
        NULL                                                                    AS CLT_REAL_EST_EFF_INS_DT,
        NULL                                                                    AS CLT_REAL_EST_EXP_INS_DT,
        NULL                                                                    AS CLT_REAL_EST_FND_PTY_IND,
        NULL                                                                    AS CLT_REAL_EST_INS_CNCL_DT,
        NULL                                                                    AS CLT_REAL_EST_INS_CO_NM,
        NULL                                                                    AS CLT_REAL_EST_INS_CVR_PCT,
        NULL                                                                    AS CLT_REAL_EST_INS_IND,
        COALESCE(CLT_REAL_EST_EVAL_TYPE, '?')                                   AS CLT_REAL_EST_LTST_EVAL_IDNX,
        COALESCE(CLT_REAL_EST_LTST_VAL_AMT_RAW, CLT_ORIG_VAL_AMT)              AS CLT_REAL_EST_LTST_VAL_AMT,
        'EUR'                                                                   AS CLT_REAL_EST_LTST_VAL_CCY_CODE,
        'ΕΥΡΩ'                                                                  AS CLT_REAL_EST_LTST_VAL_CCY_DSC,
        CLT_REAL_EST_LTST_VAL_DT,
        '-100'                                                                  AS CLT_REAL_EST_LTST_VAL_TP_CODE,
        COALESCE(CLT_REAL_EST_EVAL_TYPE, '?')                                   AS CLT_REAL_EST_LTST_VAL_TP_DSC,
        NULL                                                                    AS CLT_REAL_EST_NML_PTY_VAL,
        NULL                                                                    AS CLT_REAL_EST_NML_PTY_VAL_CCY_CODE,
        NULL                                                                    AS CLT_REAL_EST_NML_PTY_VAL_CCY_DSC,
        CLT_REAL_EST_OWN_DSC,
        '-100'                                                                  AS CLT_REAL_EST_OWN_TP_CODE,
        CLT_REAL_EST_OWNER_CDI_CODE,
        NULL                                                                    AS CLT_REAL_EST_PREFTL_CLM_AMT,
        CLT_REAL_EST_PST_CODE,
        CLT_REAL_EST_RGON,
        NULL                                                                    AS CLT_REAL_EST_RGST_NUM,
        CLT_REAL_EST_RLZN_VAL,
        NULL                                                                    AS CLT_REAL_EST_RPLCMT_VAL,
        -100                                                                    AS CLT_REAL_EST_SRC_REAL_PTY_TP_CODE,
        CLT_REAL_EST_SRC_REAL_PTY_TP_DSC,
        -100                                                                    AS CLT_REAL_EST_TP_CODE,
        CLT_REAL_EST_TP_DSC,
        CLT_REAL_EST_USE_CODE                                                   AS CLT_REAL_EST_USE_TP_CODE,
        CLT_REAL_EST_USE_DSC                                                    AS CLT_REAL_EST_USE_TP_DSC,
        CLT_REAL_EST_VAL_AGNT_RAW                                               AS CLT_REAL_EST_VAL_AGNT,
        CLT_REAL_EST_VAL_CODE_RAW                                               AS CLT_REAL_EST_VAL_CODE,
        CAST(PTY_SZ_TOTAL AS STRING)                                            AS PTY_SZ_TOTAL,
        IND_PEL
    FROM ranked_re
    WHERE rnk = 1
""")

write_npv_table("v_npv_clt_real_est", "npv_clt_real_est")

In [0]:
# =============================================================================
# Step 6 - NPV_CLT_REAL_EST_REFR + NPV_CLT_REL
# REFR: collateral registration ranks per property.
# CLT_REL: exposure-to-collateral relationships (active + closed loans).
# Output: {target_schema}.npv_clt_real_est_refr / npv_clt_rel
# =============================================================================

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_npv_clt_real_est_refr AS
    WITH ranked AS (
        SELECT 
            c.InitialLegalOrder                                                 AS CLT_REAL_EST_RGST_RANK,
            c.CollateralOldKey                                                  AS CLT_REAL_EST_CLT_CODE,
            'THS'                                                               AS CLT_REAL_EST_CLT_SRC_STM_CODE,
            c.CollateralCode                                                    AS CLT_REAL_EST_CODE,
            'THS'                                                               AS CLT_REAL_EST_SRC_STM_CODE,
            NULL                                                                AS CLT_REAL_EST_REFR_REMAN_PTY_VAL,
            re.IND_PEL,
            ROW_NUMBER() OVER (
                PARTITION BY c.CollateralOldKey, c.CollateralCode 
                ORDER BY c.InitialLegalOrder ASC
            ) AS rnk
        FROM collateral c
        JOIN v_re_perimeter re ON re.RealEstateKey = c.RealEstateKey AND re.rnk = 1
        WHERE c.RealEstateKey IS NOT NULL
          AND c.InitialLegalOrder IS NOT NULL
    )
    SELECT 
        CLT_REAL_EST_RGST_RANK,
        CLT_REAL_EST_CLT_CODE,
        CLT_REAL_EST_CLT_SRC_STM_CODE,
        CLT_REAL_EST_CODE,
        CLT_REAL_EST_SRC_STM_CODE,
        CLT_REAL_EST_REFR_REMAN_PTY_VAL,
        IND_PEL
    FROM ranked
    WHERE rnk = 1
""")

spark.sql("""
    CREATE OR REPLACE TEMPORARY VIEW v_npv_clt_rel AS
    WITH base AS (
        SELECT
            c.CollateralId                                                      AS CLT_REL_CLT_ID,
            e.ExposureSourceSystemKey                                           AS CLT_REL_AC_ID,
            e.AgreementKey                                                      AS CLT_REL_CTR_ID,
            e.ExposureKey                                                       AS CLT_REL_AC_CODE,
            'PRF'                                                               AS CLT_REL_AC_SRC_STM_CODE,
            c.CollateralOldKey                                                  AS CLT_REL_CLT_CODE,
            'THS'                                                               AS CLT_REL_CLT_SRC_STM_CODE,
            e.AgreementKey                                                      AS CLT_REL_CTR_CODE,
            'PRF'                                                               AS CLT_REL_CTR_SRC_STM_CODE,
            c.CollateralAmount                                                  AS CTR_CLT_AMT,
            COALESCE(pf.IndPel, 'UNKW')                                         AS IND_PEL,
            ROW_NUMBER() OVER (
                PARTITION BY e.ExposureKey, c.CollateralOldKey
                ORDER BY c.CollateralAmount DESC NULLS LAST, c.CollateralId
            ) AS rnk
        FROM all_exposures e
        JOIN collateral c 
            ON c.AgreementKey = e.AgreementKey
        JOIN (
            SELECT DISTINCT AgreementNumber, IndPel,
                   RANK() OVER (PARTITION BY AgreementNumber ORDER BY IndPel) AS rnk
            FROM v_perimeter_final
        ) pf ON pf.AgreementNumber = e.AgreementKey AND pf.rnk = 1
        WHERE e.PrimaryPartyFlag = true
          AND EXISTS (SELECT 1 FROM v_perimeter_final p WHERE p.AgreementNumber = e.AgreementKey)
    )
    SELECT
        CLT_REL_CLT_ID,
        CLT_REL_AC_ID,
        CLT_REL_CTR_ID,
        CLT_REL_AC_CODE,
        CLT_REL_AC_SRC_STM_CODE,
        CLT_REL_CLT_CODE,
        CLT_REL_CLT_SRC_STM_CODE,
        CLT_REL_CTR_CODE,
        CLT_REL_CTR_SRC_STM_CODE,
        CTR_CLT_AMT,
        IND_PEL
    FROM base
    WHERE rnk = 1
""")

write_npv_table("v_npv_clt_real_est_refr", "npv_clt_real_est_refr")
write_npv_table("v_npv_clt_rel", "npv_clt_rel")